In [39]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List

# Import Classiq library
from classiq import *



In [40]:
# Define the 4-qubit Heisenberg Hamiltonian
N_QUBITS = 4
pauli_terms = []
for i in range(N_QUBITS - 1):
    # Term: X_i X_{i+1}
    x_term_paulis = [Pauli.I] * N_QUBITS
    x_term_paulis[i] = Pauli.X
    x_term_paulis[i+1] = Pauli.X
    pauli_terms.append(PauliTerm(x_term_paulis, 1.0))
    
    # Term: Y_i Y_{i+1}
    y_term_paulis = [Pauli.I] * N_QUBITS
    y_term_paulis[i] = Pauli.Y
    y_term_paulis[i+1] = Pauli.Y
    pauli_terms.append(PauliTerm(y_term_paulis, 1.0))

    # Term: Z_i Z_{i+1}
    z_term_paulis = [Pauli.I] * N_QUBITS
    z_term_paulis[i] = Pauli.Z
    z_term_paulis[i+1] = Pauli.Z
    pauli_terms.append(PauliTerm(z_term_paulis, 1.0))

HAMILTONIAN = QConstant("HAMILTONIAN", List[PauliTerm], pauli_terms)

# --- Calculate Exact Ground State Energy ---
def get_exact_heisenberg_energy(n_qubits, pauli_terms):
    """Calculates the exact ground state energy using classical diagonalization."""
    sx = np.array([[0, 1], [1, 0]])
    sy = np.array([[0, -1j], [1j, 0]])
    sz = np.array([[1, 0], [0, -1]])
    identity = np.identity(2)
    
    pauli_matrices = {'X': sx, 'Y': sy, 'Z': sz, 'I': identity}
    
    ham_matrix = np.zeros((2**n_qubits, 2**n_qubits), dtype=complex)
    
    for term in pauli_terms:
        op_list = [pauli_matrices[p.name] for p in term.pauli]
        term_matrix = op_list[0]
        for op_idx in range(1, n_qubits):
            term_matrix = np.kron(term_matrix, op_list[op_idx])
        ham_matrix += term.coefficient * term_matrix
        
    eigenvalues = np.linalg.eigvalsh(ham_matrix)
    return np.min(eigenvalues)

EXACT_ENERGY = get_exact_heisenberg_energy(N_QUBITS, pauli_terms)
print(f"Exact Ground State Energy for N=4 Heisenberg Model: {EXACT_ENERGY.real:.6f}")



Exact Ground State Energy for N=4 Heisenberg Model: -6.464102


In [41]:
@qfunc
def n_block(theta: CReal, q1: QBit, q2: QBit):
    RZ(theta=-np.pi / 2, target=q2)
    CX(q1, q2)
    RZ(theta=-2 * theta, target=q1)
    RZ(theta=np.pi / 2, target=q1)
    RY(theta=2 * theta, target=q2)
    RY(theta=-np.pi / 2, target=q2)
    CX(q2, q1)
    RY(theta=-2 * theta, target=q2)
    RY(theta=np.pi / 2, target=q2)
    CX(q1, q2)
    RZ(theta=np.pi / 2, target=q1)

@qfunc
def sz_conserving_ansatz(params: CArray[CReal, 4], q: Output[QArray[QBit]]):
    allocate(N_QUBITS, q)
    X(q[1])
    X(q[3])
    n_block(params[0], q[0], q[1])
    n_block(params[1], q[2], q[3])
    n_block(params[2], q[1], q[2])
    n_block(params[3], q[0], q[3])


In [42]:
def generate_hw_efficient_ansatz(num_layers: int):
    num_params = num_layers * N_QUBITS
    @qfunc
    def hw_efficient_ansatz(params: CArray[CReal, num_params], q: Output[QArray[QBit]]):
        allocate(N_QUBITS, q)
        param_idx = 0
        for layer in range(num_layers):
            for i in range(N_QUBITS):
                RY(theta=params[param_idx], target=q[i])
                param_idx += 1
            for i in range(N_QUBITS - 1):
                CX(q[i], q[i+1])
    return hw_efficient_ansatz


In [47]:
def run_vqe(ansatz_model, hamiltonian, optimizer_settings, description):
    print(f"Running VQE for: {description}")
    
    # This classical function defines the VQE execution flow. It does not take arguments.
    @cfunc
    def vqe_main():
        res = vqe(
            hamiltonian=hamiltonian,
            maximize=False,
            initial_point=[],
            optimizer=optimizer_settings["optimizer"],
            max_iteration=optimizer_settings["max_iteration"],
            tolerance=optimizer_settings["tolerance"],
        )
        # Save the entire result object into a dictionary named "result"
        save({"result": res})

    qmod = create_model(ansatz_model, classical_execution_function=vqe_main)
    
    # Execute the model
    result_list = execute(qmod).result()
    
    # Extract the saved dictionary from the execution results
    saved_output = result_list[0].value
    
    # The VQE result object is inside the dictionary
    vqe_result_object = saved_output["result"]
    
    # The result object has an 'energy' attribute
    final_energy = vqe_result_object.energy
    
    print(f"Finished. Final Energy = {final_energy:.6f}\n")
    return final_energy

optimizer_settings = {
    "optimizer": Optimizer.COBYLA,
    "max_iteration": 150,
    "tolerance": 1e-4,
}

results = {}


In [50]:
energy_sz = run_vqe(
    sz_conserving_ansatz,
    HAMILTONIAN,
    optimizer_settings,
    "Sz-Conserving Ansatz (2 layers)"
)
results["sz_conserving"] = energy_sz

hw_layers = list(range(5, 16))
results["hw_efficient"] = {}

for layers in hw_layers:
    hw_ansatz_model = generate_hw_efficient_ansatz(layers)
    energy_hw = run_vqe(
        hw_ansatz_model,
        HAMILTONIAN,
        optimizer_settings,
        f"Hardware-Efficient Ansatz ({layers} layers)"
    )
    results["hw_efficient"][layers] = energy_hw


Running VQE for: Hardware-Efficient Ansatz (5 layers)


ClassiqError: The entry point function must be named 'main', got 'hw_efficient_ansatz'
If you need further assistance, please reach out on our Community Slack channel at: https://short.classiq.io/join-slack or open a support ticket at: https://classiq-community.freshdesk.com/support/tickets/new